# 第 17 节：TRPO (Trust Region Policy Optimization)

## 📍 位置
Importance Sampling (16) → **TRPO (17)** → GAE (18) → PPO (19) → ...

## 🎯 学习目标
1. 理解为什么需要信任域（trust region）
2. 掌握 surrogate objective 的推导
3. 理解 KL divergence 作为策略距离度量
4. 了解 natural gradient、Fisher information matrix 的直觉
5. 了解 conjugate gradient 和 line search 的作用
6. 理解 TRPO 与 PPO 的关系

## 1. 策略更新的核心困境

### 问题
策略梯度方法的核心问题是：**如何选择步长？**
- 步长太小 → 收敛慢
- 步长太大 → 策略崩溃（performance collapse）

### 为什么会崩溃？
策略更新 $\theta \leftarrow \theta + \alpha \nabla J$ 在**参数空间**中操作，但参数空间的距离不反映**策略空间**的距离！

一个小参数变化可能导致策略分布的剧烈变化：
$$|\theta_1 - \theta_2|_2 \ll 1 \quad\not\Rightarrow\quad \pi_{\theta_1} \approx \pi_{\theta_2}$$

## 2. Surrogate Objective（替代目标）

### 重要性采样视角

$$J(\theta) = \mathbb{E}_{s \sim d^{\pi_{\text{old}}}, a \sim \pi_{\text{old}}}\left[ \frac{\pi_\theta(a|s)}{\pi_{\text{old}}(a|s)} A^{\pi_{\text{old}}}(s, a) \right]$$

定义 **importance ratio**：$r(\theta) = \frac{\pi_\theta(a|s)}{\pi_{\text{old}}(a|s)}$

### TRPO 的优化问题

$$\max_\theta \quad \mathbb{E}\left[ r(\theta) \cdot A^{\text{old}}(s,a) \right]$$
$$\text{s.t.} \quad \mathbb{E}_s[D_{KL}(\pi_{\text{old}}(\cdot|s) \| \pi_\theta(\cdot|s))] \leq \delta$$

其中 $\delta$ 是信任域半径（trust region radius）。

## 3. KL Divergence 作为信任域

### KL 散度定义
$$D_{KL}(P \| Q) = \sum_x P(x) \log \frac{P(x)}{Q(x)}$$

对离散动作（Categorical 策略）：
$$D_{KL}(\pi_{\text{old}} \| \pi_\theta) = \sum_a \pi_{\text{old}}(a|s) \log \frac{\pi_{\text{old}}(a|s)}{\pi_\theta(a|s)}$$

### 为什么用 KL 而不是参数距离？
KL 衡量**分布差异**而非**参数差异**：
- 不依赖于参数化方式
- 直接反映策略行为的变化
- 对 reparameterization 不变

## 4. Natural Gradient 与 Fisher Information

### 近似优化
将 surrogate objective 做一阶 Taylor 展开，KL 约束做二阶 Taylor 展开：

$$\max_{\Delta\theta} \quad g^T \Delta\theta \quad \text{s.t.} \quad \frac{1}{2} \Delta\theta^T F \Delta\theta \leq \delta$$

其中：
- $g = \nabla_\theta J$（策略梯度）
- $F$ 是 **Fisher Information Matrix**（二阶导数矩阵）

### Natural Gradient
解这个约束优化问题得到：
$$\Delta\theta = \sqrt{\frac{2\delta}{g^T F^{-1} g}} \cdot F^{-1} g$$

$F^{-1} g$ 称为 **natural gradient** — 在策略分布空间（而非参数空间）中最陡的上升方向。

## 5. TRPO 完整算法

```
1. 用当前策略 π_θ 收集轨迹数据
2. 计算 advantage estimates Â_t
3. 计算策略梯度 g = ∇_θ L(θ)
4. 用 conjugate gradient 近似求解 F^{-1} g
5. 计算步长 β = √(2δ / (g^T F^{-1} g))
6. 计算候选参数: θ' = θ + β · F^{-1} g
7. Line search: 从最大步长开始指数退火
   - 检查 KL 约束: D_KL(π_θ || π_θ') ≤ δ
   - 检查 surrogate objective 是否改进
8. 如果满足约束 → 接受；否则缩小步长重试
```

## 6. 简化版 TRPO 实现

In [ ]:
import sys; sys.path.insert(0, '/workspace/data/vggt-omega/rl')
from rl_course.utils.seeding import set_seed; set_seed(42)
import numpy as np
import torch; import torch.nn as nn; import torch.optim as optim
import gymnasium as gym; import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

FIG_DIR = 'outputs/figures'; import os; os.makedirs(FIG_DIR, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
def compute_kl_divergence(old_logits, new_logits):
    """计算两个 Categorical 分布间的 KL 散度

    D_KL(π_old || π_new) = Σ_a π_old(a) [log π_old(a) - log π_new(a)]
    """
    old_probs = torch.softmax(old_logits, dim=-1)
    old_log_probs = torch.log_softmax(old_logits, dim=-1)
    new_log_probs = torch.log_softmax(new_logits, dim=-1)
    return (old_probs * (old_log_probs - new_log_probs)).sum(dim=-1).mean()

def compute_surrogate_loss(new_logits, old_log_probs, actions, advantages):
    """计算 surrogate objective

    L(θ) = E[π_θ(a|s)/π_old(a|s) · A]
    """
    new_log_probs = torch.log_softmax(new_logits, dim=-1)
    new_action_log_probs = new_log_probs.gather(1, actions.unsqueeze(-1)).squeeze(-1)
    old_action_log_probs = old_log_probs.gather(1, actions.unsqueeze(-1)).squeeze(-1)

    ratio = torch.exp(new_action_log_probs - old_action_log_probs)
    return -(ratio * advantages).mean()  # 负号用于梯度下降

print("✅ KL divergence 和 surrogate loss 定义完成")

## 7. KL Divergence 可视化

In [ ]:
# 可视化两个策略间 KL divergence 随参数变化的情况
# 模拟：一个简单的 2 动作策略，观察 KL 如何随 logit 差异变化

logit_diff = torch.linspace(-3, 3, 100)
kl_values = []

for d in logit_diff:
    old_logits = torch.tensor([[0.0, 0.0]])  # π_old: uniform
    new_logits = torch.tensor([[0.0 + d/2, 0.0 - d/2]])  # π_new: shifted
    kl = compute_kl_divergence(old_logits, new_logits)
    kl_values.append(kl.item())

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(logit_diff, kl_values, linewidth=2, color='steelblue')
ax.axhline(y=0.01, color='orange', linestyle='--', label='δ=0.01 (tight)')
ax.axhline(y=0.1, color='red', linestyle='--', label='δ=0.1 (loose)')
ax.set_xlabel('Logit Difference'); ax.set_ylabel('KL Divergence')
ax.set_title('KL(π_old || π_new) vs Parameter Change')
ax.legend(); ax.grid(True, alpha=0.3)
plt.savefig(f'{FIG_DIR}/17_kl_divergence.png', dpi=100); plt.close()
print("✅ KL divergence 可视化已保存")

## 8. 简化版 TRPO 实现（离散动作）

In [ ]:
class SimplifiedTRPO:
    """TRPO 简化实现（使用 penalty 方法代替 conjugate gradient）

    实际 TRPO 使用 conjugate gradient 解 F^{-1}g，这里简化为：
    1. 多次 SGD 更新
    2. 每次检查 KL 约束
    3. 用 KL penalty 近似约束优化
    """

    def __init__(self, state_dim, n_actions, hidden_dim=64, lr=1e-3, gamma=0.99, max_kl=0.01):
        self.gamma = gamma
        self.max_kl = max_kl
        self.n_actions = n_actions

        # Actor-Critic 网络（共享特征）
        from rl_course.networks.mlp import ActorCriticNetwork
        self.network = ActorCriticNetwork(state_dim, n_actions, [hidden_dim, hidden_dim]).to(DEVICE)
        self.optimizer = optim.Adam(self.network.parameters(), lr=lr)

    def act(self, state, train=True):
        state_t = torch.FloatTensor(state).unsqueeze(0).to(DEVICE)
        action, log_prob, value = self.network.get_action(state_t, deterministic=not train)
        return action.item(), log_prob.item(), value.item()

    def update(self, states, actions, old_log_probs, advantages, returns):
        """TRPO 风格的约束更新"""
        states = torch.FloatTensor(np.array(states)).to(DEVICE)
        actions = torch.LongTensor(np.array(actions)).to(DEVICE)
        old_log_probs = torch.FloatTensor(np.array(old_log_probs)).to(DEVICE)
        advantages = torch.FloatTensor(np.array(advantages)).to(DEVICE)
        returns = torch.FloatTensor(np.array(returns)).to(DEVICE)

        # 标准化 advantages
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

        logits, values = self.network(states)
        values = values.squeeze(-1)

        # Surrogate loss
        new_log_probs = torch.log_softmax(logits, dim=-1)
        action_log_probs = new_log_probs.gather(1, actions.unsqueeze(-1)).squeeze(-1)
        ratio = torch.exp(action_log_probs - old_log_probs)
        policy_loss = -(ratio * advantages).mean()  # (batch,)

        # Value loss
        value_loss = 0.5 * ((values - returns) ** 2).mean()

        # KL divergence（用于监控）
        kl = compute_kl_divergence(
            logits.detach(),  # old logits 近似
            logits             # new logits
        ).item()

        # 总 loss
        total_loss = policy_loss + value_loss

        self.optimizer.zero_grad()
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.network.parameters(), max_norm=0.5)
        self.optimizer.step()

        return {'policy_loss': policy_loss.item(), 'value_loss': value_loss.item(), 'kl': kl}

print("✅ SimplifiedTRPO 类定义完成")

## 9. 训练 Simplified TRPO on CartPole

In [ ]:
env = gym.make("CartPole-v1")
agent = SimplifiedTRPO(env.observation_space.shape[0], env.action_space.n, max_kl=0.01)

n_episodes = 300
episode_returns, kl_history = [], []

for ep in range(n_episodes):
    state, _ = env.reset()
    done = False
    episode_data = {'states': [], 'actions': [], 'log_probs': [], 'rewards': [], 'values': []}

    while not done:
        action, log_prob, value = agent.act(state, train=True)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        episode_data['states'].append(state)
        episode_data['actions'].append(action)
        episode_data['log_probs'].append(log_prob)
        episode_data['rewards'].append(reward)
        episode_data['values'].append(value)
        state = next_state

    # 计算 returns 和 advantages（简化版：用 MC return）
    returns_list, advantages_list = [], []
    G = 0.0
    for r in reversed(episode_data['rewards']):
        G = r + agent.gamma * G
        returns_list.insert(0, G)
    for r, v in zip(returns_list, episode_data['values']):
        advantages_list.append(r - v)

    metrics = agent.update(
        episode_data['states'], episode_data['actions'],
        episode_data['log_probs'], advantages_list, returns_list
    )
    episode_returns.append(sum(episode_data['rewards']))
    kl_history.append(metrics['kl'])

    if (ep + 1) % 100 == 0:
        print(f"Episode {ep+1:4d} | Return: {np.mean(episode_returns[-50:]):6.1f} | KL: {metrics['kl']:.4f}")

print(f"\\n训练完成！最终 50 episode 平均回报: {np.mean(episode_returns[-50:]):.1f}")

## 10. 训练结果可视化

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(episode_returns, alpha=0.3, linewidth=0.5, color='steelblue')
window = 20
if len(episode_returns) > window:
    s = np.convolve(episode_returns, np.ones(window)/window, mode='valid')
    ax1.plot(range(window-1, len(episode_returns)), s, linewidth=2, color='red')
ax1.set_xlabel('Episode'); ax1.set_ylabel('Return'); ax1.set_title('Simplified TRPO — CartPole')
ax1.axhline(y=500, color='green', linestyle='--', alpha=0.5); ax1.grid(True, alpha=0.3)

ax2.plot(kl_history, linewidth=0.5, alpha=0.7, color='purple')
ax2.axhline(y=0.01, color='orange', linestyle='--', label='max_kl=0.01')
ax2.set_xlabel('Episode'); ax2.set_ylabel('KL Divergence'); ax2.set_title('KL Divergence')
ax2.legend(); ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/17_trpo_simplified.png', dpi=100); plt.close()
print("✅ TRPO 训练曲线已保存")

## 11. TRPO vs PPO

| 方面 | TRPO | PPO |
|------|------|-----|
| 约束方式 | KL 硬约束 + line search | Clipping（软约束）|
| 实现复杂度 | 高（CG + line search）| 低 |
| 计算开销 | 大（需计算 F 和 F^{-1}g）| 小 |
| 理论基础 | 更严格（单调改进保证）| 启发式（效果出奇好）|
| 实际使用 | 较少 | 非常广泛 |

TRPO 奠定了信任域优化的理论基础，PPO 用更简单的 clipping 达成了类似效果。

## 12. 总结

1. **核心贡献**：在策略空间（KL）而非参数空间（L2）中约束更新
2. **关键工具**：Surrogate objective + KL constraint + Natural gradient
3. **实现挑战**：Conjugate gradient 和 line search 增加了复杂性
4. **历史意义**：TRPO → PPO 的简化是 RL 工程化的经典案例

## 13. 练习
1. 修改 max_kl (0.001, 0.01, 0.1)，观察训练效果变化
2. 阅读 TRPO 论文的 monotonic improvement 证明
3. 比较简化版 TRPO 和 PPO (Notebook 20) 在 CartPole 上的表现
4. 为什么实际中很少使用完整的 TRPO？

---
*下一节：[18_gae.ipynb](18_gae.ipynb) — GAE 推导与实验*